# 28 · 心跳离线段检测：间隔阈值与时长排名

> **难度** ★★☆ · **领域** IoT · **前置** 案例 13、18
>
> **涉及 API**：`sort` / `shift(1, partition_by=)` / `filter` / `derive` / `rank().over(partition_by=, order_by=, descending=)` / `group_by().agg()` + `count_if` / `max_if`

**学习目标**

- 把"心跳超时"翻译成有序计算：**相邻间隔 > 阈值 ⇒ 一次离线事件**——离线
  区间的起止就藏在心跳行里
- 用 `rank().over()` 给离线事件按时长排 severity，取每设备的**最严重一次**
- 处理"无离线设备"的 NULL 聚合结果——`max_if` 无满足行返回 NULL 的语义

## 问题描述

设备心跳 `hb(device, ts)`，ts 为分钟（每分钟应有一条）。规则：**相邻心跳
间隔超过 5 分钟 ⇒ 中间发生一次离线**，离线区间为 (上次心跳, 本次心跳)。
目标：

1. 找出所有离线区间（起、止、时长）；
2. 给每设备的离线按时长排 severity，输出**最严重一次**；
3. 汇总每设备的离线次数与最长时长。

预判：设备 A 心跳 0, 3, 6, 15, 18, 60, 63 → 离线 (6,15) 9 分钟、(18,60)
42 分钟；设备 B 0, 2, 4, 7 → 无离线。

In [ ]:
from ltseq import LTSeq

hb = LTSeq.from_rows([
    {"device": "A", "ts": 0},
    {"device": "A", "ts": 3},
    {"device": "A", "ts": 6},
    {"device": "A", "ts": 15},
    {"device": "A", "ts": 18},
    {"device": "A", "ts": 60},
    {"device": "A", "ts": 63},
    {"device": "B", "ts": 0},
    {"device": "B", "ts": 2},
    {"device": "B", "ts": 4},
    {"device": "B", "ts": 7},
]).sort("device", "ts")

hb.show(11);

> **预期输出**
>
> ```text
> +--------+----+
> | device | ts |
> +--------+----+
> | A      | 0  |
> | A      | 3  |
> | A      | 6  |
> | A      | 15 |
> | A      | 18 |
> | A      | 60 |
> | A      | 63 |
> | B      | 0  |
> | B      | 2  |
> | B      | 4  |
> | B      | 7  |
> +--------+----+
> ```

## 第一步 · 间隔：组内 shift 与差值

老三样：组内 `shift(1)` 拿上次心跳，差值即间隔。首行 prev 为 NULL，gap
随之 NULL——首行天然不是离线事件，NULL 语义又一次自动正确。

In [ ]:
gapped = hb.derive(
    prev=lambda r: r.ts.shift(1, partition_by="device"),
).derive(
    gap=lambda r: r.ts - r.prev,
)

gapped.show(11);

> **预期输出**
>
> ```text
> +--------+----+------+------+
> | device | ts | prev | gap  |
> +--------+----+------+------+
> | A      | 0  | None | None |
> | A      | 3  | 0    | 3    |
> | A      | 6  | 3    | 3    |
> | A      | 15 | 6    | 9    |
> | A      | 18 | 15   | 3    |
> | A      | 60 | 18   | 42   |
> | A      | 63 | 60   | 3    |
> | B      | 0  | None | None |
> | B      | 2  | 0    | 2    |
> | B      | 4  | 2    | 2    |
> | B      | 7  | 4    | 3    |
> +--------+----+------+------+
> ```

**解析**

- 每个离线区间**无需单独构造**：它的起点是上一行 ts、终点是本行 ts——
  间隔超阈值的行本身就携带区间的全部信息。这是"事件即区间"的视角转换，
  省掉了配对或展开。
- gap 为 NULL 的行（每设备首行）在下一步 `gap > 5` 时为 NULL，不被
  filter 选中——首行不会被误判为离线，无需特判。
- 若心跳是时间戳列，差值改用
  `r.ts.dt.diff(r.prev, unit="minute")`——思路不变，单位显式化。

## 第二步 · 离线区间与 severity 排名

filter 出间隔超 5 的行，补上区间的起止 (start, end)（`gap` 就是分钟
时长），再按设备内时长降序排名——
`rank().over(partition_by=, order_by=, descending=)` 一次到位。

In [ ]:
from ltseq import rank

offline = gapped.filter(lambda r: r.gap > 5).derive(
    start=lambda r: r.prev,
    end=lambda r: r.ts,
    severity=lambda r: rank().over(
        partition_by=r.device, order_by=r.gap, descending=True
    ),
)

offline.sort("device", "ts").show();

> **预期输出**
>
> ```text
> +--------+----+------+-----+-------+-----+----------+
> | device | ts | prev | gap | start | end | severity |
> +--------+----+------+-----+-------+-----+----------+
> | A      | 15 | 6    | 9   | 6     | 15  | 2        |
> | A      | 60 | 18   | 42  | 18    | 60  | 1        |
> +--------+----+------+-----+-------+-----+----------+
> ```

**解析**

- `.over(order_by=...)` 目前会让结果行按窗口键（gap 降序）排列，所以
  展示前按时间序排回来。
- B 设备不在结果里：它的最大间隔只有 3，`gap > 5` 全军覆没——filter 的
  空结果不是错误，是"无离线"的正确表达（本案例的对照组）。
- **一条 derive 三个新列**：start/end 引用已有列，severity 是独立窗口
  表达式——互不引用即可同行，与"新列不能互相引用"的规则并不矛盾。
- `rank()` 并列同名次：若 A 出现两次 42 分钟离线，severity=1 会同时命中
  两段——与案例 16 的"并列全保"口径一致。
- severity 语义是"设备内最严重"；要**全局**最严重，去掉 partition_by
  即可——窗口参数的一字之差，报表口径天壤之别。

## 第三步 · 最严重事件与设备汇总

severity == 1 的行就是每设备的"最长离线"；设备级汇总用条件聚合
`count_if` / `max_if` 一次算完。

In [ ]:
worst = offline.filter(lambda r: r.severity == 1)

worst.show();

> **预期输出**
>
> ```text
> +--------+----+------+-----+-------+-----+----------+
> | device | ts | prev | gap | start | end | severity |
> +--------+----+------+-----+-------+-----+----------+
> | A      | 60 | 18   | 42  | 18    | 60  | 1        |
> +--------+----+------+-----+-------+-----+----------+
> ```

**解析**

- 排名先行，筛选殿后：`filter(severity == 1)` 是"组内 Top 1"的通用收尾，
  比 `group_by + max` 再 join 回原表少了两次传递。
- 若还要看"第二长"，`severity <= 2`——排名列天然支持任意 Top-K。

In [ ]:
from ltseq import count_if, max_if

per_device = (
    gapped
    .group_by("device")
    .agg(
        outages=lambda g: count_if(g.gap > 5),
        worst_gap=lambda g: max_if(g.gap > 5, g.gap),
    )
    .sort("device")
)

per_device.show();

> **预期输出**
>
> ```text
> +--------+---------+-----------+
> | device | outages | worst_gap |
> +--------+---------+-----------+
> | A      | 2       | 42        |
> | B      | 0       | None      |
> +--------+---------+-----------+
> ```

**解析**

- `max_if(g.gap > 5, g.gap)`：只对满足条件的行取 max——B 没有任何满足行，
  返回 **NULL**。NULL 在这里是"无离线"的准确表达；报表要显示 0 时，
  `derive(worst=lambda r: nvl(r.worst_gap, 0))` 一行兜底。
- `count_if` 与 `max_if` 的谓词都引用列表达式，和普通聚合并排写——条件
  聚合家族（`count_if/sum_if/avg_if/min_if/max_if`）是告警报表的高频
  组件。
- 守恒检查：per_device 的 outages 之和（2+0）应等于 offline 的行数（2）✔
  ——两条路线互为印证，是数据管道的标准自检。

## 关键点解析

- **事件即区间**：离线区间不需要展开构造——"间隔超阈值"的行自带起止。
  同类视角：超时请求（发出↔回复）、库存断档（卖出↔补货）。
- **NULL 的三连击**：shift 边界 NULL → gap NULL → filter 不选中——首行
  的"无前值"被三值逻辑层层正确处理，一行特判都没写。
- **rank 的 partition_by 是报表口径**：设备内排名（本案例）还是全局排名
  （去掉 partition_by），写窗口表达式时永远先问"在谁的范围内比"。

## 延伸练习

1. 设备重启的陷阱：若设备重启后时钟归零，按 ts 排序会把重启前后的心跳
   交错在一起，所有间隔看上去都"正常"。改按到达顺序排（加一列 `seq`，
   `sort("device", "seq")`），让归零显现为负的 gap，再在 derive 里加
   `gap < 0` 的标记列，并决定它该算离线、丢数还是异常。
2. 离线时刻的"上下文"：若心跳表还带有其他字段（固件版本、电量），用
   `on=lambda a, b: (a.device == b.device) & (a.start == b.ts)` 把 offline
   与心跳原表 `join`，给每次离线附上离线前最后一条心跳的字段——方便回查
   日志。
3. 按小时统计离线时长：对 offline 的 (start, end) 做与案例 23 相反的
   "拆分"操作（提示：给 offline 和一张小时槽表加上同一个常量键列并按它
   `join`，让每次离线与每个小时配对；保留有重叠的配对，再用 `if_else`
   把区间裁剪到该小时——体会区间拆分的难度）。
4. 把阈值 5 参数化：LTSeq 的 lambda 能捕获外部变量——写一个
   `def offline_report(hb, threshold): ...` 返回离线表，验证闭包变量在
   表达式 lambda 里的可用性。

---
**上一例**：[27 · 重采样与日历对齐](27_resample_align.ipynb) · **回[目录](../README.md)**